# I. Impact environnemental de l’alimentation
- Comparer les impacts environnementaux des grands groupes d’aliments
- Statistiques descriptives + visualisation
- Comparer en particulier les produits animaux et végétaux
- Identifier les principaux postes responsables de l’impact climatique : agriculture, transformation, emballage, transport...

# II. Les alternatives végétales : vraiment meilleures ?
- Identifier les principales catégories d’alternatives végétales
- Comparer leurs impacts environnementaux avec ceux des produits animaux qu’elles peuvent remplacer
- Comparer leur profil nutritionnel et étudier colonne SAIN et LIM
- Voir si certaines alternatives végétales présentent effectivement un compromis entre impact environnemental et qualité nutritionnelle
# III. Santé et environnement : existe-t-il un double gain ?
- Chercher les aliments qui combinent faible impact environnemental et bonne qualité nutritionnelle
- Comparer notamment les profils des aliments animaux, végétaux et alternatives végétales
- Conclure sur l’existence ou non d’un arbitrage santé–environnement, et sur les aliments qui semblent permettre un double gain

In [ ]:
# Chargement de la base de données fusionnée
import pandas as pd
database = pd.read_csv("database.csv")

print("Dimensions de la base :", database.shape)
database.head(5)

La base de données fusionnée contient 2296 lignes et 34 colonnes. 

I. Impact environnemental de l'alimentation

Dans cette première partie, nous analysons les impacts environnementaux des différents groupes alimentaires. 

I.1. Comparaison des impacts environnementaux des grands groupes alimentaires

In [ ]:
# Nombre de groupes et de sous-groupes alimentaires
print("Nombre de groupes alimentaires :", database["groupe_aliment"].nunique())
print("Nombre de sous-groupes alimentaires :", database["sous_groupe_aliment"].nunique())

# Nombre de sous-groupes et de produits par groupe alimentaire
composition_groupes = database.groupby("groupe_aliment").agg(
    nombre_sous_groupes=("sous_groupe_aliment", "nunique"),
    nombre_produits=("nom_produit", "count")
)

# Affichage du tableau
display(composition_groupes.sort_values("nombre_produits", ascending=False))

La base de données distingue 11 groupes alimentaires : viandes, œufs, poissons ; fruits, légumes, légumineuses et oléagineux ; produits céréaliers ; entrées et plats composés ; lait et produits laitiers ; aides culinaires et ingrédients divers ; boissons ; produits sucrés ; aliments infantiles ; matières grasses ; glaces et sorbets. Le groupe « viandes, œufs, poissons » est le plus représenté, avec 12 sous-groupes et 637 produits.

In [ ]:
# Comparaison des impacts environnementaux des grands groupes d’aliments

# Liste des indicateurs environnementaux disponibles dans la base fusionnée
indicateurs = [
    "score_unique_ef",
    "changement_climatique",
    "utilisation_du_sol",
    "epuisement_ressources_eau",
    "particules_fines",
    "toxicite_humaine_non_cancerogenes",
    "toxicite_humaine_cancerogenes"
]

# Calcul des statistiques environnementales par groupe alimentaire
stats_groupes = database.groupby("groupe_aliment").agg(
    nombre_produits=("nom_produit", "count"),
    **{
        indicateur + "_moyenne": (indicateur, "mean")
        for indicateur in indicateurs
    },
    **{
        indicateur + "_mediane": (indicateur, "median")
        for indicateur in indicateurs
    }
)

# Affichage des résultats
display(stats_groupes)

In [ ]:
# Recherche des groupes ayant les impacts les plus élevés

for indicateur in indicateurs:
    print(indicateur)
    print("Moyenne maximale :", stats_groupes[indicateur + "_moyenne"].idxmax())
    print("Médiane maximale :", stats_groupes[indicateur + "_mediane"].idxmax())

Le groupe viandes, œufs et poissons présente la moyenne et la médiane les plus élevées pour 5 indicateurs sur 7. Il possède notamment le score unique EF moyen le plus élevé (1,64) et l’impact climatique moyen le plus élevé (13,18 kg CO₂ eq/kg), ainsi que les médianes les plus élevées pour ces deux indicateurs (1,27 et 7,75 kg CO₂ eq/kg). Les résultats diffèrent pour l'utilisation des sols : les viandes, œufs et poissons ont la moyenne la plus élevée, mais les matières grasses ont la médiane la plus élevée. L'épuisement des ressources en eau présente un résultat différent : les matières grasses ont la moyenne la plus élevée, tandis que les aides culinaires et ingrédients divers ont la médiane la plus élevée.

In [ ]:
# Recherche des groupes ayant les impacts les plus faibles

for indicateur in indicateurs:
    print(indicateur)
    print("Moyenne minimale :", stats_groupes[indicateur + "_moyenne"].idxmin())
    print("Médiane minimale :", stats_groupes[indicateur + "_mediane"].idxmin())

Les boissons ont globalement les impacts environnementaux les plus faibles parmi les 11 groupes alimentaires, avec notamment un score unique EF médian de 0,117 et un impact climatique médian de 0,83 kg CO₂ eq/kg.

Le graphique ci-dessous présente la médiane de l’indicateur « changement climatique » pour chaque groupe alimentaire.

In [ ]:
# Graphique sur l'impact climatique médian des groupes alimentaires

import matplotlib.pyplot as plt

impact_climatique_groupes = (
    database.groupby("groupe_aliment")["changement_climatique"]
    .median()
    .sort_values()
)

plt.figure(figsize=(10, 6))

impact_climatique_groupes.plot(kind="barh")

plt.xlabel("Changement climatique (kg CO₂ eq)")
plt.ylabel("Groupe alimentaire")
plt.title("Impact climatique médian selon le groupe alimentaire")

plt.tight_layout()
plt.show()

Ce graphique montre que le groupe « viandes, œufs, poissons » présente l'impact climatique médian le plus élevé parmi les groupes alimentaires étudiés. 

Après avoir identifié les groupes alimentaires présentant les impacts environnementaux les plus élevés, notamment en matière de changement climatique, il convient maintenant de déterminer quelles étapes du cycle de vie des produits contribuent le plus à leur impact climatique (agriculture, transformation, emballage, transport, distribution et consommation).

I.2. Identification des principaux postes responsables de l’impact climatique

In [ ]:
# Identification des principaux postes du cycle de vie responsables de l’impact climatique

postes_climatiques = [
    "changement_climatique_agriculture",
    "changement_climatique_transformation",
    "changement_climatique_emballage",
    "changement_climatique_transport",
    "changement_climatique_distribution",
    "changement_climatique_consommation"
]

# Calcul de la moyenne et de la médiane pour chaque poste
stats_postes = database[postes_climatiques].agg(["mean", "median"]).T

# Renommage des colonnes
stats_postes.columns = ["moyenne", "mediane"]

# Classement des postes selon leur impact moyen 
stats_postes = stats_postes.sort_values(
    "moyenne", ascending=False
)

# Affichage du tableau final
display(stats_postes)

Les résultats montrent que l'agriculture constitue le principal contributeur à l'impact climatique des produits alimentaires étudiés, avec un impact moyen d'environ 4,73 kg CO₂ eq par kg de produit. 

In [ ]:
# Identification des principaux postes du cycle de vie responsables de l’impact climatique par groupe alimentaire

# Calcul de la moyenne pour chaque poste et chaque groupe
moyennes_postes_groupes = database.groupby("groupe_aliment")[postes_climatiques].mean()

# Affichage des résultats
display(moyennes_postes_groupes)

L'agriculture est le principal poste contributeur dans les 11 groupes alimentaires. 

Après avoir identifié le groupe « viandes, œufs et poissons » comme présentant les impacts environnementaux les plus élevés, intéressons-nous aux différences entre ses sous-groupes.

I.3. Hétérogénéité des impacts au sein du groupe « viandes, œufs et poissons »

In [ ]:
# Sélection du groupe viandes, œufs et poissons
animaux = database[
    database["groupe_aliment"] == "viandes, œufs, poissons"
]

# Calcul des statistiques descriptives par sous-groupe
stats_sous_groupes = animaux.groupby("sous_groupe_aliment").agg(
    nombre_produits=("nom_produit", "count"),
    **{
        indicateur + "_moyenne": (indicateur, "mean")
        for indicateur in indicateurs
    },
    **{
        indicateur + "_mediane": (indicateur, "median")
        for indicateur in indicateurs
    }
)

# Affichage du tableau
display(stats_sous_groupes)

In [ ]:
# Recherche des sous-groupes ayant les impacts les plus élevés
for indicateur in indicateurs:
    print("\n", indicateur)
    print("Moyenne maximale :", stats_sous_groupes[indicateur + "_moyenne"].idxmax())
    print("Médiane maximale :", stats_sous_groupes[indicateur + "_mediane"].idxmax())

On constate que les viandes cuites présentent les médianes les plus élevées pour les sept indicateurs environnementaux étudiés.

In [ ]:
# Recherche des sous-groupes ayant les impacts les plus faibles
for indicateur in indicateurs:
    print("\n", indicateur)
    print("Moyenne minimale :", stats_sous_groupes[indicateur + "_moyenne"].idxmin())
    print("Médiane minimale :", stats_sous_groupes[indicateur + "_mediane"].idxmin())

Les substituts de viande présentent globalement les impacts les plus faibles parmi les sous-groupes étudiés.

Si l'on se concentre sur l’impact climatique et le score unique EF global, on constate que les impacts environnementaux sont très hétérogènes au sein du groupe viandes, œufs et poissons. 
Les viandes cuites et crues présentent les impacts climatiques médians les plus élevés, tandis que les œufs et surtout les substituts de viande et de charcuterie présentent des impacts nettement plus faibles. 
Par exemple, la médiane de l'impact climatique des viandes cuites est de 29,70 kg CO₂ eq/kg de produit, contre 2,02 kg CO₂ eq/kg pour les substituts de viande. De plus, les viandes cuites présentent un score unique EF médian de 2,82, contre 0,244 pour les substituts de viande. Ce score est un indicateur synthétique qui regroupe plusieurs impacts environnementaux en un seul indicateur. 
Cependant, ces sous-groupes sont représentés par des effectifs très différents dans la base : les viandes cuites comptent 104 produits et les viandes crues 132, contre 3 produits pour les substituts de charcuterie et 8 pour les substituts de viande. 

In [ ]:
# Graphique sur l'impact climatique médian des sous-groupes
# du groupe "viandes, œufs et poissons"

impact_climatique_sous_groupes = (
    database[
        database["groupe_aliment"] == "viandes, œufs, poissons"
    ]
    .groupby("sous_groupe_aliment")["changement_climatique"]
    .median()
    .sort_values()
)

plt.figure(figsize=(10, 7))

impact_climatique_sous_groupes.plot(kind="barh")

plt.xlabel("Changement climatique (kg CO₂ eq)")
plt.ylabel("Sous-groupe alimentaire")
plt.title(
    "Impact climatique médian des sous-groupes "
    "de viandes, œufs et poissons"
)

plt.tight_layout()
plt.show()

Ce graphique permet de visualiser les impacts climatiques médians des différents sous-groupes. Il met en évidence l'impact élevé des viandes cuites.